# MLS Feature Sampling

Sample point features at arbitrary query locations with local linear MLS.

In [ ]:
import torch
import torchbvh as tb

N, M, D, C = 32, 12, 3, 8
points = torch.randn(N, D, device="cuda")
x = torch.randn(N, C, device="cuda", requires_grad=True)

# Random query locations inside the point cloud bounding box.
lo = points.min(dim=0).values
hi = points.max(dim=0).values
query = lo + torch.rand(M, D, device="cuda") * (hi - lo)

`BVH.interpolate(...)` is useful when you want to build the BVH once and reuse it for one or more interpolation calls.

In [ ]:
with tb.BVH(points) as bvh:
    sampled, field_grad = bvh.interpolate(query, x, k=4, return_grad=True)

print(sampled.shape)     # (M, C)
print(field_grad.shape)  # (M, D, C)

loss = sampled.square().mean()
loss.backward()
print(x.grad.shape)

`tb.mls_interpolate(...)` is the one-shot functional form. It builds the BVH internally, runs the query, and returns the interpolated features.

In [ ]:
sampled_again = tb.mls_interpolate(points, query, x.detach(), k=4)

print(sampled_again.shape)  # (M, C)

For batched point clouds, pass `(B, N, D)` points, `(B, M, D)` query locations, and `(B, N, C)` features.

In [ ]:
B, N, M, D, C = 4, 32, 12, 3, 8
batched_points = torch.randn(B, N, D, device="cuda")
batched_x = torch.randn(B, N, C, device="cuda")

lo = batched_points.min(dim=1).values[:, None, :]
hi = batched_points.max(dim=1).values[:, None, :]
batched_query = lo + torch.rand(B, M, D, device="cuda") * (hi - lo)

batched_sampled = tb.mls_interpolate(batched_points, batched_query, batched_x, k=4)

print(batched_sampled.shape)  # (B, M, C)